# Optimize Iceberg
Compact tables, retain recent snapshots, and sweep unreachable files.

In [ ]:
catalog = {"name": "rekep", "properties": {}}
namespace = None
branch = "root"
min_files = 2
retain = 24
snapshot_age_days = 7
orphan_age_days = 3
remove_orphans = True
metadata = True
log_level = "INFO"

In [ ]:
import datetime

from rekep.iceberg import IcebergCatalog
from rekep.logs import Stage, configure

configure(log_level)

if type(retain) is not int or retain < 1:
    raise ValueError("retain must be a positive integer")
if type(min_files) is not int or min_files < 2:
    raise ValueError("min_files must be at least 2")
if snapshot_age_days is not None and snapshot_age_days < 0:
    raise ValueError("snapshot_age_days must be non-negative or null")
if orphan_age_days < 0:
    raise ValueError("orphan_age_days must be non-negative")

store = IcebergCatalog.from_dict(catalog)
stage = Stage("optimize_iceberg", sources={"catalog": store.name})
snapshot_age = (
    None if snapshot_age_days is None else datetime.timedelta(days=snapshot_age_days)
)
orphan_age = datetime.timedelta(days=orphan_age_days)

In [ ]:
reports = {}
for dataset in store.datasets(namespace):
    reports[dataset.name] = dataset.optimize(
        branch=branch,
        min_files=min_files,
        retain=retain,
        older_than=snapshot_age,
        remove_orphans=remove_orphans,
        orphan_age=orphan_age,
        metadata=metadata,
    )

rewritten = sum(report["rewritten"] for report in reports.values())
deleted = sum(report["deleted"] for report in reports.values())
byte_size = sum(report["bytes"] for report in reports.values())
stage.says(
    "visited %d tables: %d parts compacted, %d snapshots expired, %d files swept (%d bytes)",
    len(reports),
    rewritten,
    sum(report["expired"] for report in reports.values()),
    deleted,
    byte_size,
)
# A maintenance pass reads every table it visits and writes the parts it
# compacted, which is what `read` and `written` mean everywhere else.
result = stage.finished(
    read=len(reports),
    written=rewritten,
    skipped=len(reports) - sum(1 for report in reports.values() if report["rewritten"]),
    tables=len(reports),
    expired=sum(report["expired"] for report in reports.values()),
    deleted=deleted,
    byte_size=byte_size,
    reports=reports,
)
result